# Phase 3 Quick Eval — TN Sanity Check

**Purpose**: 1 seed × 20 episodes on χ=16 (worst) and χ=128 (best) to check whether TN compression produces usable L1 errors before committing to the full 3-seed × 200-episode sweep.

**Expected runtime**: ~30-40 min on Kaggle P100.

**Go/No-Go**: compare L1 error against Phase 1 baseline (0.2849 ± 0.0030). If either χ is catastrophically worse (>200% Δ), flag for healing/fine-tune before full sweep.

In [ ]:
# ── Cell 1: Install / pin dependencies ───────────────────────────────────────
# Mirrors Phase 1 environment: SM detection, torch==2.2.2+cu118 for P100 (sm_60),
# JAX/Flax removal, versioned NCCL stub.
import subprocess, sys, os, ctypes, re, glob as _glob

def _pip(*args, check=True):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=check)

# ── 1. Detect GPU SM version BEFORE any torch import ─────────────────────────
_sm_ver = 70
try:
    _smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15,
    )
    if _smi.returncode == 0:
        _cap = _smi.stdout.strip().split('\n')[0].strip()
        _sm_ver = int(_cap.replace('.', ''))
        print(f'nvidia-smi: compute_cap={_cap}  (sm_{_sm_ver})')
except Exception as _e:
    print(f'nvidia-smi check skipped ({_e}) — assuming sm_70+')

# ── 2. Install ALL non-torch deps FIRST ──────────────────────────────────────
print('Installing non-torch deps ...')
_pip(
    'transformers==4.40.1',
    'tokenizers==0.19.1',
    'timm==0.9.10',
    'sentencepiece==0.1.99',
)
_pip(
    'triton==2.3.0',         # bnb 0.43.3 needs triton 2.x; 2.3.0 is earliest cp312 wheel
    'accelerate>=0.27.0',
    'bitsandbytes==0.43.3',  # 0.49+ requires torch>=2.3
    'pynvml',
    'PyYAML',
    'tqdm',
    'av',
    'datasets',
    'matplotlib',
)
print('Non-torch deps installed.')

# ── 3. Pin SM-compatible torch LAST ──────────────────────────────────────────
if _sm_ver < 70:
    print(f'sm_{_sm_ver} < 70 → pinning torch==2.2.2+cu118 + torchvision==0.17.2+cu118 ...')
    _pip(
        'torch==2.2.2+cu118',
        'torchvision==0.17.2+cu118',  # system 0.25.0 calls torch.library.register_fake (torch 2.4+)
        '--index-url', 'https://download.pytorch.org/whl/cu118',
    )
    print('torch==2.2.2+cu118 + torchvision==0.17.2+cu118 pinned.')
else:
    print('sm_70+ → using default Kaggle torch.')

# ── 4. Uninstall JAX/Flax ────────────────────────────────────────────────────
print('Uninstalling jax/jaxlib/flax ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax', 'jaxlib', 'flax'],
    check=False,
)
_np_diag = subprocess.run(
    [sys.executable, '-c',
     'import numpy; print("numpy", numpy.__version__, numpy.__file__)'],
    capture_output=True, text=True, timeout=30,
)
print(_np_diag.stdout.strip() or _np_diag.stderr.strip())

# ── 5. NCCL stub (sm_60 only) ─────────────────────────────────────────────────
# torch==2.2.2+cu118 links against nccl* symbols; no matching libnccl.so exists
# on the Kaggle P100 runtime.  Compile a zero-stub and load RTLD_GLOBAL.
if _sm_ver < 70:

    def _find_libtorch_cuda():
        for _pat in [
            '/usr/local/lib/python*/dist-packages/torch/lib/libtorch_cuda.so',
            '/opt/conda/lib/python*/site-packages/torch/lib/libtorch_cuda.so',
        ]:
            _hits = _glob.glob(_pat)
            if _hits:
                return _hits[0]
        raise FileNotFoundError('libtorch_cuda.so not found — torch install failed')

    def _nccl_undef_syms(path):
        r = subprocess.run(['objdump', '-T', path],
                           capture_output=True, text=True, timeout=60)
        syms = {}
        for line in r.stdout.splitlines():
            if '*UND*' not in line:
                continue
            m = re.search(r'\(([\w.]+)\)\s+(nccl\w+)', line)
            if m:
                syms[m.group(2)] = m.group(1)
                continue
            m2 = re.search(r'\*UND\*\s+\S+\s+(NCCL_[\d.]+)\s+(nccl\w+)', line)
            if m2:
                syms[m2.group(2)] = m2.group(1)
                continue
            m3 = re.search(r'\s+(nccl\w+)\s*$', line)
            if m3:
                syms.setdefault(m3.group(1), None)
        if not syms:
            r2 = subprocess.run(['nm', '-D', path],
                                capture_output=True, text=True, timeout=60)
            for line in r2.stdout.splitlines():
                parts = line.split()
                if len(parts) >= 2 and parts[-2] == 'U' and parts[-1].startswith('nccl'):
                    syms[parts[-1]] = None
        return syms

    _libtorch_cuda = _find_libtorch_cuda()
    print(f'Scanning NCCL symbols in: {_libtorch_cuda}')
    _nccl_syms = _nccl_undef_syms(_libtorch_cuda)
    print(f'Found {len(_nccl_syms)} undefined nccl* symbols')

    _c_lines = ['#include <stddef.h>', '']
    for _sym, _ver in sorted(_nccl_syms.items()):
        _internal = f'_stub_{_sym}'
        _c_lines.append(f'int {_internal}(void) {{ return 0; }}')
        if _ver:
            _c_lines.append(f'__asm__(".symver {_internal},{_sym}@@{_ver}");')
        else:
            _c_lines.append(
                f'extern int {_sym}(void) __attribute__((alias("{_internal}")));'
            )
        _c_lines.append('')

    _stub_c  = '/tmp/_nccl_stub.c'
    _stub_so = '/tmp/_nccl_stub.so'
    with open(_stub_c, 'w') as _f:
        _f.write('\n'.join(_c_lines))

    _gcc = subprocess.run(
        ['gcc', '-shared', '-fPIC', '-nostartfiles', '-o', _stub_so, _stub_c],
        capture_output=True, text=True, timeout=30,
    )
    if _gcc.returncode != 0:
        print(f'GCC stderr:\n{_gcc.stderr.strip()}')
        raise RuntimeError('NCCL stub compilation failed')

    ctypes.CDLL(_stub_so, mode=ctypes.RTLD_GLOBAL)
    print(f'NCCL stub loaded (RTLD_GLOBAL): {_stub_so}')

    import torch as _torch_smoke
    print(f'torch {_torch_smoke.__version__} imported OK ✓')
    _cuda_ok = _torch_smoke.cuda.is_available()
    _dev = _torch_smoke.cuda.get_device_name(0) if _cuda_ok else 'N/A'
    print(f'CUDA: available={_cuda_ok}, device={_dev}')
    del _torch_smoke

print('Cell 1 complete.')

In [ ]:
# ── Cell 2: Verify transformers version ───────────────────────────────────────
# Fails loudly before the 15 GB model download if the wrong version landed.
# If this cell raises: Runtime -> Disconnect and delete runtime, then re-run
# from Cell 1 without running anything else first.
import transformers

REQUIRED_TRANSFORMERS = "4.40.1"
_ver = transformers.__version__
print(f"transformers : {_ver}")

if _ver != REQUIRED_TRANSFORMERS:
    raise RuntimeError(
        f"transformers {_ver} installed but OpenVLA requires exactly {REQUIRED_TRANSFORMERS}.\n"
        f"Other 4.x builds and all 5.x builds break modeling_prismatic.py\n"
        f"(_supports_sdpa missing, is_flax_available moved, etc.).\n\n"
        f"Fix:\n"
        f"  1. Runtime -> Disconnect and delete runtime\n"
        f"  2. Re-open the notebook and run Cell 1 before anything else\n"
        f"  3. Do NOT run any other cells before Cell 1 finishes"
    )
print(f"  confirmed: {REQUIRED_TRANSFORMERS}")

In [ ]:
# ── Cell 3: Clone repo and install the vlam_compress package ─────────────────
import os, sys, subprocess

IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB  = False
try:
    import google.colab; IS_COLAB = True
except ImportError:
    pass

REPO_DIR = "/kaggle/working/vlam" if IS_KAGGLE else "/content/vlam"

_gh_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _gh_token = UserSecretsClient().get_secret("GH_TOKEN")
        print("GH_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Kaggle Secrets ({_e}).")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _gh_token = userdata.get("GH_TOKEN")
        print("GH_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Colab Secrets ({_e}).")
else:
    _gh_token = os.environ.get("GH_TOKEN", "")

if _gh_token:
    REPO_URL = f"https://{_gh_token}@github.com/quantumadventurer11/vw-quantum-vlam-challenge"
else:
    print("No GH_TOKEN — using unauthenticated URL (public repo).")
    REPO_URL = "https://github.com/quantumadventurer11/vw-quantum-vlam-challenge"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
# --no-deps: prevents pyproject.toml torch>=2.2.0 from upgrading our pinned torch
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
# Editable installs create a .pth file that Kaggle's running kernel may not re-read
_src_path = os.path.join(REPO_DIR, "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
print(f"Working directory : {os.getcwd()}")
print(f"Environment       : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local'}")
print(f"sys.path[0]: {sys.path[0]}")
print("vlam_compress package installed.")

In [ ]:
# ── Cell 4: HuggingFace authentication ───────────────────────────────────────
# openvla/openvla-7b downloads without auth in practice (Phase 1 confirmed this).
from huggingface_hub import login

_hf_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Kaggle Secrets ({_e}) — proceeding without auth.")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Colab Secrets ({_e}).")
else:
    import os as _os; _hf_token = _os.environ.get("HF_TOKEN", "")

if _hf_token:
    login(token=_hf_token, add_to_git_credential=False)
else:
    print("No HF_TOKEN — model will attempt unauthenticated download.")

In [ ]:
import json
import platform
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")   # headless backend for Colab
import matplotlib.pyplot as plt
import numpy as np
import pynvml
import torch
import torch.nn as nn
import transformers
import yaml
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

from vlam_compress.compress import (
    mps_decompose, mps_reconstruct, count_core_params,
    find_compression_targets, DEFAULT_TARGET_SUFFIXES,
)
from vlam_compress.metrics import (
    aggregate, aggregate_runs,
    efficiency_gain_pct, param_efficiency_gain_pct,
    model_compression_ratio, co2e_grams, delta_dict, flag_benchmark,
)

print(f"PyTorch        : {torch.__version__}")
print(f"Transformers   : {transformers.__version__}")
print("All imports OK.")

In [ ]:
assert torch.cuda.is_available(), "No GPU — change runtime to GPU and restart."
props = torch.cuda.get_device_properties(0)
print(f"GPU      : {props.name}")
print(f"VRAM     : {props.total_memory / 1024**3:.1f} GB")
print(f"CUDA     : {torch.version.cuda}")
print(f"Platform : {platform.platform()}")

In [ ]:
# ── Cell 5: Locate Phase 2 checkpoints ────────────────────────────────────────
# On Kaggle: Phase 2 kernel output is mounted as a kernel_source at
#   /kaggle/input/vw-phase2-compression/
# On Colab (fallback): Google Drive mount.
from pathlib import Path
import os

if os.path.isdir("/kaggle/input/vw-phase2-compression/checkpoints"):
    CHECKPOINTS_BASE = Path("/kaggle/input/vw-phase2-compression/checkpoints")
    print(f"Kaggle kernel-source: {CHECKPOINTS_BASE}")
elif os.path.isdir("/kaggle/input/vw-phase2-compression"):
    CHECKPOINTS_BASE = Path("/kaggle/input/vw-phase2-compression")
    print(f"Kaggle kernel-source (root): {CHECKPOINTS_BASE}")
else:
    # Colab fallback: try Google Drive
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINTS_BASE = Path("/content/drive/MyDrive/vlam_checkpoints")
        print("Google Drive mounted.")
    except Exception as _e:
        CHECKPOINTS_BASE = Path("checkpoints")
        print(f"Local fallback: {CHECKPOINTS_BASE}  ({_e})")

print(f"Checkpoints dir  : {CHECKPOINTS_BASE}")


In [ ]:
# ── Quick-eval config (sanity check: 1 seed × 20 episodes × χ∈{16,128}) ──────
with open("configs/seeds.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Sanity-check overrides ────────────────────────────────────────────────────
SEEDS           = [42]          # single seed
N_EVAL_EPISODES = 20            # 20 episodes per run
ALL_BOND_DIMS   = [16, 128]     # worst-case + best-case only
# ─────────────────────────────────────────────────────────────────────────────

MODEL_ID        = "openvla/openvla-7b"
UNNORM_KEY      = "toto"
HF_DATASET_ID   = "lerobot/toto"
CHECKPOINTS_DIR = CHECKPOINTS_BASE
RESULTS_DIR     = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

VISION_KEYWORDS = ("vision", "patch_embed", "visual", "vit", "image_encoder")

CHI_SWEEP = [chi for chi in ALL_BOND_DIMS
             if (CHECKPOINTS_DIR / f"compressed_chi{chi}" / "cores.pt").exists()]

print(f"Seeds           : {SEEDS}")
print(f"Episodes/run    : {N_EVAL_EPISODES}")
print(f"Dataset         : {HF_DATASET_ID}")
print(f"χ to evaluate   : {CHI_SWEEP}")
if not CHI_SWEEP:
    raise RuntimeError(f"No cores.pt found in {CHECKPOINTS_DIR}.")


In [ ]:
# ── Load Phase 1 baseline metrics ─────────────────────────────────────────────
baseline_path = RESULTS_DIR / "baseline_metrics.json"
assert baseline_path.exists(), (
    f"Phase 1 baseline not found at {baseline_path}. "
    "Commit results/baseline_metrics.json from Phase 1 first."
)
with open(baseline_path) as f:
    baseline_data = json.load(f)

B_AGG = baseline_data["aggregate"]
N_PARAMS_TOTAL = baseline_data["hardware"]["n_params_total"]

# Normalize key names: Phase 1 uses "l1_error" / "inference_time_ms";
# Phase 3 internal code uses "l1_error_mean" / "inference_time_ms_mean".
# Adding aliases here makes both access patterns work without KeyError.
if "l1_error" in B_AGG and "l1_error_mean" not in B_AGG:
    B_AGG["l1_error_mean"] = B_AGG["l1_error"]
if "inference_time_ms" in B_AGG and "inference_time_ms_mean" not in B_AGG:
    B_AGG["inference_time_ms_mean"] = B_AGG["inference_time_ms"]

print(f"Baseline (INT8-only):")
print(f"  L1 error   : {B_AGG['l1_error']['mean']:.4f} ± {B_AGG['l1_error']['std']:.4f}")
print(f"  Infer time : {B_AGG['inference_time_ms']['mean']:.1f} ± "
      f"{B_AGG['inference_time_ms']['std']:.1f} ms")
print(f"  Total params: {N_PARAMS_TOTAL/1e9:.3f}B")

# ── Load Phase 2 sweep stats ──────────────────────────────────────────────────
sweep_stats_path = RESULTS_DIR / "compression_sweep_stats.json"
if sweep_stats_path.exists():
    with open(sweep_stats_path) as f:
        sweep_data = json.load(f)
    N_TARGET_PARAMS = sweep_data["n_target_params_orig"]
    N_NONTARGET     = N_PARAMS_TOTAL - N_TARGET_PARAMS
    CHI_CORE_COUNTS = {
        int(k): v["total_core_params"]
        for k, v in sweep_data["sweep_stats"].items()
    }
    print(f"\nPhase 2 sweep stats loaded.")
    print(f"  Target layer params : {N_TARGET_PARAMS/1e9:.3f}B")
    print(f"  Non-target (fixed)  : {N_NONTARGET/1e9:.3f}B")
    for chi, n_core in sorted(CHI_CORE_COUNTS.items()):
        ratio = model_compression_ratio(N_PARAMS_TOTAL, N_NONTARGET, n_core)
        print(f"  χ={chi:3d}: {n_core/1e6:.0f}M core params  →  {ratio:.2f}× whole-model ratio")
else:
    print("⚠  compression_sweep_stats.json not found — compression ratios will be estimated.")
    CHI_CORE_COUNTS = {}
    N_TARGET_PARAMS = None
    N_NONTARGET = None

In [ ]:
pynvml.nvmlInit()
gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
driver_ver = pynvml.nvmlSystemGetDriverVersion()
if isinstance(driver_ver, bytes):
    driver_ver = driver_ver.decode()
print(f"pynvml ready — GPU: {pynvml.nvmlDeviceGetName(gpu_handle)}, driver: {driver_ver}")

In [ ]:
# ── Dataset helpers (identical to Phase 1 — HuggingFace Hub / LeRobot format)
task_map: dict = {}  # populated by load-dataset cell


def hf_item_to_sample(item: dict) -> dict:
    # Returns dict with keys: image (PIL), language (str), action_gt (np.ndarray shape 7)
    import av, io

    # ── Image ─────────────────────────────────────────────────────────────────
    img = None
    for _key in ("observation.image", "observation.images.image_0",
                 "observation.images.image", "image", "rgb"):
        if _key in item:
            img = item[_key]
            break
    if img is None:
        for _key in item:
            if "image" in _key.lower():
                img = item[_key]
                break
    if img is None:
        raise KeyError(f"No image key found. Available: {list(item.keys())}")

    if isinstance(img, dict):
        _path  = img.get("path")
        _bytes = img.get("bytes")
        if _path is not None:
            _ts = float(img.get("timestamp") or 0.0)
            with av.open(_path) as _c:
                _stream = _c.streams.video[0]
                if _ts > 0:
                    _pts = int(_ts / float(_stream.time_base))
                    _c.seek(_pts, stream=_stream)
                for _f in _c.decode(video=0):
                    img = Image.fromarray(_f.to_ndarray(format="rgb24"))
                    break
        elif _bytes is not None:
            img = Image.open(io.BytesIO(_bytes))
        else:
            raise TypeError(f"Unrecognised image dict keys: {list(img.keys())}")
    elif isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype(np.uint8))

    if not isinstance(img, Image.Image):
        raise TypeError(f"Could not decode image (got {type(img).__name__})")

    # ── Action ────────────────────────────────────────────────────────────────
    action = item.get("action")
    if action is None:
        raise KeyError(f"'action' not in item. Available: {list(item.keys())}")
    if isinstance(action, dict):
        _parts = [np.array(v, dtype=np.float32).ravel() for v in action.values()]
        action_gt = np.concatenate(_parts)[:7]
    else:
        action_gt = np.array(action, dtype=np.float32).ravel()[:7]

    # ── Language instruction ──────────────────────────────────────────────────
    lang = (item.get("language_instruction")
            or item.get("task")
            or item.get("instruction"))
    if lang is None:
        _tidx = item.get("task_index")
        lang = task_map.get(_tidx, "") if _tidx is not None else ""
    if isinstance(lang, (bytes, bytearray)):
        lang = lang.decode("utf-8")
    lang = str(lang).strip() or "pick up the object"

    return {"image": img, "language": lang, "action_gt": action_gt}


print("Dataset helper defined.")

In [ ]:
# ── Load bridge_dataset from HuggingFace Hub ──────────────────────────────────
# Uses the lerobot/bridge dataset mirror (LeRobot v2.0 format).
# No Google Cloud credentials needed — data streams from HuggingFace Hub.
import json as _json
from huggingface_hub import hf_hub_download

print(f"Loading {HF_DATASET_ID} from HuggingFace Hub (streaming mode)...")
hf_ds = load_dataset(HF_DATASET_ID, split="train", streaming=True)

try:
    _tasks_file = hf_hub_download(HF_DATASET_ID, filename="meta/tasks.jsonl", repo_type="dataset")
    with open(_tasks_file) as _f:
        for _line in _f:
            _d = _json.loads(_line.strip())
            task_map[_d["task_index"]] = _d["task"]
    print(f"Loaded {len(task_map)} tasks from meta/tasks.jsonl")
except Exception as _e:
    print(f"Could not load tasks.jsonl ({_e}) — generic language fallback active.")

_peek = next(iter(hf_ds))
print(f"\nDataset columns ({len(_peek)}):")
for _col, _val in _peek.items():
    _dtype = type(_val).__name__
    if isinstance(_val, (list, tuple)):
        _dtype = f"list[{len(_val)}]"
    elif isinstance(_val, dict):
        _dtype = f"dict{{{', '.join(_val.keys())}}}"
    print(f"  {_col}: {_dtype}")
del _peek

print("\nDataset ready — frames streamed on demand per seed.")

In [ ]:
# ── Load base model in FP16 ──────────────────────────────────────────────────
# bitsandbytes INT8 (load_in_8bit=True) does not support sm_60 (P100).
# We use FP16 instead, consistent with the Phase 1 baseline.
from transformers import AutoModelForVision2Seq, AutoProcessor

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model in FP16 (may take 10-20 min on first run)...")
torch.cuda.reset_peak_memory_stats()

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    attn_implementation="eager",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

peak_mem_load_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)
print(f"Model loaded in FP16. Peak GPU memory at load: {peak_mem_load_mib:.0f} MiB")


In [ ]:
# ── Inference and evaluation helpers ─────────────────────────────────────────

def run_inference_timed(image, language):
    """Single timed forward pass. Returns (action_pred, ms, peak_mib, avg_pwr_w)."""
    torch.cuda.reset_peak_memory_stats()
    inputs = processor(language, image).to("cuda:0")

    pwr_before = pynvml.nvmlDeviceGetPowerUsage(gpu_handle) / 1000.0
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.no_grad():
        action_pred = model.predict_action(**inputs, unnorm_key=UNNORM_KEY, do_sample=False)

    torch.cuda.synchronize()
    t1 = time.perf_counter()
    pwr_after = pynvml.nvmlDeviceGetPowerUsage(gpu_handle) / 1000.0

    return (
        action_pred,
        (t1 - t0) * 1000.0,
        torch.cuda.max_memory_allocated() / 1024**2,
        (pwr_before + pwr_after) / 2.0,
    )


def run_one_seed(seed, label=""):
    """
    Evaluate the current model state on N_EVAL_EPISODES episodes drawn by *seed*.
    Returns a per-seed result dict.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    ds_eval = (
        hf_ds
        .shuffle(seed=seed, buffer_size=5000)
        .take(N_EVAL_EPISODES)
    )

    l1s, ts, mems, pwrs = [], [], [], []
    skipped = 0
    wall_t0 = time.perf_counter()

    for item in tqdm(
        ds_eval, total=N_EVAL_EPISODES,
        desc=f"{label} seed={seed}", leave=False,
    ):
        try:
            sample = hf_item_to_sample(item)
        except (KeyError, TypeError, ValueError):
            skipped += 1
            continue

        pred, t_ms, mem, pwr = run_inference_timed(sample["image"], sample["language"])

        gt = sample["action_gt"]
        n  = min(len(pred), len(gt))
        l1s.append(float(np.abs(pred[:n] - gt[:n]).mean()))
        ts.append(t_ms)
        mems.append(mem)
        pwrs.append(pwr)

    wall_s  = time.perf_counter() - wall_t0
    avg_pwr = float(np.mean(pwrs)) if pwrs else 0.0
    kwh     = avg_pwr * wall_s / 3.6e6

    return {
        "seed": seed, "n_samples": len(l1s), "n_skipped": skipped,
        "l1_error_mean":          float(np.mean(l1s)),
        "l1_error_std":           float(np.std(l1s)),
        "inference_time_ms_mean": float(np.mean(ts)),
        "inference_time_ms_std":  float(np.std(ts)),
        "peak_mem_mib_mean":      float(np.mean(mems)),
        "avg_power_w":            avg_pwr,
        "wall_time_s":            round(wall_s, 2),
        "total_kwh":              round(kwh, 6),
        "co2e_g":                 round(co2e_grams(kwh), 4),
    }


def run_all_seeds(label=""):
    """Run 3 seeds and return (per_seed_list, aggregate_dict)."""
    results = [run_one_seed(s, label) for s in SEEDS]
    return results, aggregate_runs(results)


print("Inference helpers defined.")

In [ ]:
# ── TN patch / restore helpers ────────────────────────────────────────────────

def apply_tn_patches(cores_dict):
    """
    For each layer in cores_dict, reconstruct W_hat from its MPS cores and
    replace the module with a standard nn.Linear containing the reconstructed weight.

    Returns saved_modules: {layer_name: (parent_module, child_name, original_module)}
    for later restoration with restore_tn_patches().
    """
    saved = {}
    all_named = dict(model.named_modules())

    with torch.no_grad():
        for layer_name, layer_cores in cores_dict.items():
            if "." not in layer_name:
                continue
            parent_name, child_name = layer_name.rsplit(".", 1)
            parent = all_named.get(parent_name)
            if parent is None:
                continue
            orig_mod = getattr(parent, child_name, None)
            if orig_mod is None or not hasattr(orig_mod, "weight"):
                continue

            w_device = orig_mod.weight.device
            W_hat = mps_reconstruct(
                [c.to(w_device, dtype=torch.float32) for c in layer_cores],
                tuple(orig_mod.weight.shape),
            ).to(torch.float16)

            has_bias = getattr(orig_mod, "bias", None) is not None
            new_mod  = nn.Linear(
                W_hat.shape[1], W_hat.shape[0],
                bias=has_bias, device=w_device, dtype=torch.float16,
            )
            new_mod.weight = nn.Parameter(W_hat)
            if has_bias:
                new_mod.bias = nn.Parameter(orig_mod.bias.data.to(torch.float16))

            saved[layer_name] = (parent, child_name, orig_mod)
            setattr(parent, child_name, new_mod)

    return saved


def restore_tn_patches(saved):
    """Restore original modules from saved dict."""
    for _, (parent, child_name, orig_mod) in saved.items():
        setattr(parent, child_name, orig_mod)


def load_cores(chi, variant="llm_only"):
    """
    Load cores from checkpoint for the given chi.
    variant="llm_only" filters out vision-encoder layers.
    variant="full"     returns all cores in the checkpoint.
    """
    path = CHECKPOINTS_DIR / f"compressed_chi{chi}" / "cores.pt"
    if not path.exists():
        raise FileNotFoundError(path)
    all_cores = torch.load(path, map_location="cpu", weights_only=False)
    if variant == "full":
        return all_cores
    # llm_only: exclude vision layers
    return {
        k: v for k, v in all_cores.items()
        if not any(kw in k.lower() for kw in VISION_KEYWORDS)
    }


print("TN patch helpers defined.")

In [ ]:
# ── Warm-up (prime CUDA kernels before timed runs) ────────────────────────────
print("Running warm-up (3 forward passes)...")
warmup_sample = hf_item_to_sample(next(iter(hf_ds)))

wu_inputs = processor(warmup_sample["language"], warmup_sample["image"]).to("cuda:0")
with torch.no_grad():
    for _ in range(3):
        _ = model.predict_action(**wu_inputs, unnorm_key=UNNORM_KEY, do_sample=False)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
print("Warm-up complete. Memory stats reset.")

In [ ]:
# ── Sanity-check sweep: χ∈{16,128}, 1 seed × 20 episodes ────────────────────
quick_results = {}

for chi in CHI_SWEEP:
    print(f"
{'─'*60}")
    print(f"χ = {chi}  (1 seed × {N_EVAL_EPISODES} episodes)")
    print(f"{'─'*60}")
    try:
        cores = load_cores(chi, variant="llm_only")
    except FileNotFoundError as e:
        print(f"  SKIP: {e}")
        continue

    saved = apply_tn_patches(cores)
    print(f"  Patched {len(saved)} LLM layers.")

    result = run_one_seed(SEEDS[0], label=f"χ={chi}")
    restore_tn_patches(saved)
    torch.cuda.empty_cache()

    quick_results[chi] = result
    l1 = result["l1_error_mean"]
    bl1 = B_AGG["l1_error"]["mean"]
    delta = l1 - bl1
    print(f"  L1 error  : {l1:.4f} ± {result['l1_error_std']:.4f}")
    print(f"  Baseline  : {bl1:.4f}  →  Δ = {delta:+.4f}  ({100*delta/max(bl1,1e-9):+.1f}% rel)")
    print(f"  Infer time: {result['inference_time_ms_mean']:.1f} ms")

print("
" + "="*60)
print("SANITY CHECK SUMMARY")
print("="*60)
bl1 = B_AGG["l1_error"]["mean"]
print(f"Baseline (Phase 1 FP16): L1 = {bl1:.4f} ± {B_AGG['l1_error']['std']:.4f}")
print()
for chi, r in sorted(quick_results.items()):
    l1 = r["l1_error_mean"]
    delta = l1 - bl1
    pct = 100*delta/max(bl1,1e-9)
    flag = "OK" if abs(pct) < 50 else ("DEGRADED" if abs(pct) < 200 else "CATASTROPHIC")
    print(f"  χ={chi:3d}: L1 = {l1:.4f}  Δ = {delta:+.4f} ({pct:+.1f}%)  [{flag}]")
print()
print("GO/NO-GO guide:")
print("  |ΔL1| < 50% of baseline → reasonable, proceed to full sweep")
print("  |ΔL1| 50-200%           → degraded, flag to Vidhi before full sweep")
print("  |ΔL1| > 200%            → catastrophic, healing/fine-tune needed")

# Save quick results
import json as _json
_out = RESULTS_DIR / "quickeval_sanity.json"
_payload = {
    "description": "Sanity check: 1 seed x 20 episodes, chi in [16,128]",
    "baseline_l1": bl1,
    "baseline_l1_std": B_AGG["l1_error"]["std"],
    "results": {str(chi): r for chi, r in quick_results.items()},
}
with open(_out, "w") as _f:
    _json.dump(_payload, _f, indent=2)
print(f"
Saved: {_out}")
